# Heretic to ONNX on Colab

This notebook runs the full Gemma 4 Heretic -> raw ONNX -> q4f16 -> packaged browser repo pipeline.

Recommended runtime:
- GPU runtime enabled
- Paid Colab plan if available
- High-memory runtime if offered in the runtime settings


In [ ]:
import os
from pathlib import Path

# Repo source: set either REPO_URL or REPO_ZIP_IN_DRIVE.
REPO_URL = "https://github.com/lightnolimit/gemma-4-E2B-it-heretic-ara-ONNX.git"
REPO_BRANCH = "main"
REPO_ZIP_IN_DRIVE = ""

# Hugging Face model inputs.
SOURCE_MODEL_ID = "p-e-w/gemma-4-E2B-it-heretic-ara"
BASE_MODEL_ID = "google/gemma-4-E2B-it"
TARGET_REPO_ID = "lightnolimit/gemma-4-E2B-it-heretic-ara-ONNX"

# Output locations.
WORKSPACE_DIR = Path("/content/heretic")
WORK_DIR = WORKSPACE_DIR / "build/work/heretic-ara-colab"
OUTPUT_DIR = WORKSPACE_DIR / "build/heretic-ara-colab-package"
DRIVE_EXPORT_DIR = Path("/content/drive/MyDrive/heretic-onnx-output")

# Try Colab secrets first, then environment, then manual override.
HF_TOKEN = os.environ.get("HF_TOKEN", "")
try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or userdata.get("HF_TOKEN")
except Exception:
    pass

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
else:
    raise ValueError("Set HF_TOKEN in Colab secrets or environment before running this notebook.")

MANIFEST_TEMPLATE = "configs/heretic-to-onnx.gemma4-e2b-heretic-ara.yaml"
MANIFEST_RUNTIME = "configs/heretic-to-onnx.colab.runtime.yaml"


In [ ]:
import shutil
import subprocess
import zipfile

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

if WORKSPACE_DIR.exists():
    print(f"Reusing existing workspace: {WORKSPACE_DIR}")
elif REPO_URL:
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(WORKSPACE_DIR)], check=True)
elif REPO_ZIP_IN_DRIVE and Path(REPO_ZIP_IN_DRIVE).exists():
    with zipfile.ZipFile(REPO_ZIP_IN_DRIVE) as archive:
        archive.extractall("/content")
    extracted_root = WORKSPACE_DIR
    if not extracted_root.exists():
        candidates = [path for path in Path("/content").iterdir() if path.is_dir() and (path / "pyproject.toml").exists()]
        if len(candidates) == 1:
            candidates[0].rename(WORKSPACE_DIR)
    if not WORKSPACE_DIR.exists():
        raise FileNotFoundError("Repo zip extracted, but /content/heretic was not created. Rename the extracted folder or update WORKSPACE_DIR.")
else:
    raise ValueError("Set REPO_URL or REPO_ZIP_IN_DRIVE before continuing.")

print(WORKSPACE_DIR)


In [ ]:
import shutil
import subprocess
import sys

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("Switch Colab to a GPU runtime before continuing.")

gpu_info = subprocess.check_output([
    "nvidia-smi",
    "--query-gpu=name,memory.total",
    "--format=csv,noheader,nounits",
], text=True).strip().splitlines()[0]
gpu_name, gpu_memory_mb = [part.strip() for part in gpu_info.split(",", 1)]
gpu_memory_gb = int(gpu_memory_mb) / 1024
print(f"GPU: {gpu_name} | VRAM: {gpu_memory_gb:.1f} GB")

if gpu_memory_gb < 24:
    raise RuntimeError(
        f"This pipeline is not recommended on {gpu_name} with {gpu_memory_gb:.1f} GB VRAM. "
        "Use at least a 24 GB card such as L4, and preferably 40 GB+ for reliable export."
    )

subprocess.run([sys.executable, "-m", "pip", "install", "-U",
                "huggingface_hub[cli]",
                "PyYAML>=6.0",
                "accelerate>=1.0.0",
                "safetensors>=0.4.3",
                "sentencepiece>=0.2.0",
                "onnx>=1.17.0",
                "onnxruntime-gpu",
                "onnxconverter-common>=1.14.0",
                "git+https://github.com/huggingface/transformers.git"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(WORKSPACE_DIR)], check=True)
subprocess.run(["nvidia-smi"], check=True)


In [ ]:
import sys

import yaml

manifest_template_path = WORKSPACE_DIR / MANIFEST_TEMPLATE
manifest_runtime_path = WORKSPACE_DIR / MANIFEST_RUNTIME

with manifest_template_path.open("r", encoding="utf-8") as handle:
    manifest = yaml.safe_load(handle)

manifest["source_model_id"] = SOURCE_MODEL_ID
manifest["base_model_id"] = BASE_MODEL_ID
manifest["target_repo_id"] = TARGET_REPO_ID

with manifest_runtime_path.open("w", encoding="utf-8") as handle:
    yaml.safe_dump(manifest, handle, sort_keys=False)

print(manifest_runtime_path)
subprocess.run([sys.executable, "-m", "tools.heretic_to_onnx", "bootstrap"], cwd=str(WORKSPACE_DIR), check=True)


In [ ]:
convert_cmd = [
    sys.executable,
    "-m",
    "tools.heretic_to_onnx",
    "convert",
    "--config",
    str(WORKSPACE_DIR / MANIFEST_RUNTIME),
    "--work-dir",
    str(WORK_DIR),
    "--output-dir",
    str(OUTPUT_DIR),
    "--force",
    "--strict-onnx",
    "--export-mode",
    "execute",
    "--quantize-mode",
    "execute",
    "--python-exec",
    sys.executable,
]

print(" ".join(convert_cmd))
subprocess.run(convert_cmd, cwd=str(WORKSPACE_DIR), check=True)


In [ ]:
import shutil

DRIVE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
archive_path = shutil.make_archive("/content/heretic-ara-colab-package", "zip", root_dir=str(OUTPUT_DIR))
copied_path = DRIVE_EXPORT_DIR / Path(archive_path).name
shutil.copy2(archive_path, copied_path)

print("Package directory:", OUTPUT_DIR)
print("Zip archive:", archive_path)
print("Copied to Drive:", copied_path)
